# Brain Tumor MRI Classification – Transfer Learning Model (v2)

**Architecture:** EfficientNetB3 (ImageNet weights) → Global Average Pooling → BatchNorm → Dropout → Dense softmax  
**Training strategy:** Two-phase (frozen base → fine-tune top 20 layers)  
**Classes:** glioma · meningioma · notumor · pituitary



### Changes from v1 (87.1% → target 90%+)
| What changed | Why |
|---|---|
| EfficientNetB0 → **EfficientNetB3** | Stronger backbone, bigger receptive field — biggest single upgrade |
| IMG_SIZE 224 → **300×300** | B3's native resolution; B0 at 224 was undersized for B3 |
| Phase 2 LR `1e-5` → **3e-5** | Previous LR was too small for 20 unfrozen layers to learn meaningfully |
| Unfreeze top 30 → **top 20 layers** | Fewer unfrozen layers = more stable fine-tuning, less catastrophic forgetting |
| Added **RandomBrightness + RandomTranslation** augmentation | Helps meningioma recall (was 0.66) by showing more variability |
| Phase 1 patience 5 → **8** | Gives the frozen-head phase more room to converge |
| Phase 2 patience 7 → **10** | Gives fine-tuning more room; previous run stopped after only 8 epochs |
| Output folder → `BrainTumor_Results_v2` | Keeps v1 results safe for comparison |

### Overview of this notebook
​
In this notebook we implemented a transfer learning pipeline for brain tumor MRI classification using EfficientNetB3 as backbone, trained on four classes (glioma, meningioma, notumor, pituitary). The goal was to improve over a previous v1 model and compare this heavier architecture against a lighter ResNet18 baseline.

## Data loading and preprocessing
​
We mount Google Drive, point to the pre‑split train.csv, val.csv, and test.csv, and normalize Windows-style paths so they work in the Colab environment. Class distributions are close to balanced, and we compute class weights to slightly compensate for residual differences between tumor types.
​

Images are read from disk, decoded as RGB, resized to 300×300 (EfficientNetB3’s native resolution), and passed through the corresponding preprocess_input function. On the training split we apply relatively strong augmentation (flip, rotation, zoom, contrast, brightness, translation) to improve generalization, especially for meningioma where recall was weaker in v1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA_ROOT = '/content/drive/MyDrive/Brain_Tumor_Data'

# ── Verify the expected structure is there ────────────────────────────────────
for sub in ['Training', 'Testing', 'train.csv', 'val.csv', 'test.csv']:
    path = os.path.join(DRIVE_DATA_ROOT, sub)
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {status}  {path}')

# ── 📁 Output folder (auto-created) ──────────────────────────────────────────
DRIVE_OUT = '/content/drive/MyDrive/BrainTumor_Results_v2'
os.makedirs(DRIVE_OUT, exist_ok=True)
print(f'\n✅ Output folder ready: {DRIVE_OUT}')

# Helper: save figure to Drive AND show inline
def savefig(filename):
    import matplotlib.pyplot as _plt
    path = os.path.join(DRIVE_OUT, filename)
    _plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  💾 Saved → {path}')

Mounted at /content/drive
  ✅  /content/drive/MyDrive/Brain_Tumor_Data/Training
  ✅  /content/drive/MyDrive/Brain_Tumor_Data/Testing
  ✅  /content/drive/MyDrive/Brain_Tumor_Data/train.csv
  ✅  /content/drive/MyDrive/Brain_Tumor_Data/val.csv
  ✅  /content/drive/MyDrive/Brain_Tumor_Data/test.csv

✅ Output folder ready: /content/drive/MyDrive/BrainTumor_Results_v2


## 0) Imports & Configuration

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT = DRIVE_DATA_ROOT
TRAIN_CSV = os.path.join(DATA_ROOT, 'train.csv')
VAL_CSV   = os.path.join(DATA_ROOT, 'val.csv')
TEST_CSV  = os.path.join(DATA_ROOT, 'test.csv')

# Checkpoints saved directly to Drive
CKPT_PHASE1 = os.path.join(DRIVE_OUT, 'best_phase1.keras')
CKPT_BEST   = os.path.join(DRIVE_OUT, 'best_model.keras')
MODEL_FINAL = os.path.join(DRIVE_OUT, 'brain_tumor_classifier_v2.keras')

# ── Hyper-parameters ──────────────────────────────────────────────────────────
IMG_SIZE        = (300, 300)   # EfficientNetB3 native resolution (was 224 for B0)
BATCH_SIZE      = 32
EPOCHS_FROZEN   = 20           # was 15 — more room for the head to converge
EPOCHS_FINETUNE = 25           # was 20
LR_FROZEN       = 1e-3
LR_FINETUNE     = 3e-5         # was 1e-5 — previous run's fine-tune barely moved
DROPOUT         = 0.4
NUM_CLASSES     = 4
FINETUNE_LAYERS = 20           # unfreeze top 20 layers (was 30 — too many)

CLASS_NAMES  = ['glioma', 'meningioma', 'notumor', 'pituitary']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print(f'Backbone: EfficientNetB3 @ {IMG_SIZE}')

TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Backbone: EfficientNetB3 @ (300, 300)


## 1) Load CSV Splits

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)


def fix_path(p):
    p = p.replace('\\', '/')          
    if os.path.isabs(p):               
        return p
    # Strip any leading 'Brain Tumor data/' prefix if the CSV stored full relative paths
    for prefix in ['Brain Tumor data/', './Brain Tumor data/']:
        if p.startswith(prefix):
            p = p[len(prefix):]
            break
    return os.path.join(DATA_ROOT, p)

for df in [train_df, val_df, test_df]:
    df['filepath'] = df['filepath'].apply(fix_path)

# Sanity check – verify the first few files actually exist on Drive
missing = [p for p in train_df['filepath'].head(5) if not os.path.exists(p)]
if missing:
    print('⚠️  Some paths could not be found – check DRIVE_DATA_ROOT:')
    for p in missing:
        print('   ', p)
else:
    print('✅ File paths verified OK')

print(f'\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('\nClass distribution (train):')
print(train_df['label'].value_counts())

✅ File paths verified OK

Train: 4708 | Val: 1009 | Test: 1009

Class distribution (train):
label
pituitary     1218
notumor       1212
meningioma    1144
glioma        1134
Name: count, dtype: int64


## 2) Class Weights

In [ ]:
train_counts = train_df['label'].value_counts()
total = len(train_df)

class_weight = {
    CLASS_TO_IDX[cls]: total / (NUM_CLASSES * train_counts[cls])
    for cls in CLASS_NAMES
}
print('Class weights:', class_weight)

Class weights: {0: np.float64(1.0379188712522045), 1: np.float64(1.0288461538461537), 2: np.float64(0.9711221122112211), 3: np.float64(0.9663382594417077)}


## 3) tf.data Pipeline

Images are loaded from Drive, decoded to RGB, resized to 224×224, and passed through
EfficientNet's `preprocess_input`. Augmentation is applied on the training set only.

In [ ]:
# Stronger augmentation to improve meningioma recall (was 0.66 in v1)
augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1),        # new: MRI intensity variations
    layers.RandomTranslation(0.1, 0.1),  # new: slight spatial shift
], name='augmentation')


def load_and_preprocess(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = preprocess_input(img)
    return img, label


def make_dataset(df, training=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(
        (df['filepath'].values, df['label'].map(CLASS_TO_IDX).values)
    )
    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(
            lambda img, lbl: (augmentation(img, training=True), lbl),
            num_parallel_calls=tf.data.AUTOTUNE
        )
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(train_df, training=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)

print('Batches → train:', len(train_ds), '| val:', len(val_ds), '| test:', len(test_ds))

Batches → train: 148 | val: 32 | test: 32


## 4) Model Definition

### Model architecture and training strategy
​
We build an EfficientNetB3 feature extractor with ImageNet weights, remove the original classification head, and add global average pooling, batch normalization, dropout and a final softmax layer for the four tumor classes. The notebook uses a two‑phase training strategy: first training only the new head with the backbone frozen, then unfreezing and fine‑tuning the top 20 layers of the base model.
​

In Phase 1 we train with a relatively high learning rate (1e‑3) and early stopping, saving the best checkpoint on the validation set. In Phase 2 we unfreeze a small portion of the backbone (top 20 layers) and continue training with a low learning rate (3e‑5) and more patience to allow gradual adaptation without catastrophic forgetting.

In [ ]:
def build_model(trainable_base=False):
    base = EfficientNetB3(          # upgraded from B0
        include_top=False,
        weights='imagenet',
        input_shape=(*IMG_SIZE, 3)
    )
    base.trainable = trainable_base

    inputs  = keras.Input(shape=(*IMG_SIZE, 3))
    x       = base(inputs, training=trainable_base)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    return keras.Model(inputs, outputs), base


model, base_model = build_model(trainable_base=False)
model.summary()

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 10, 10, 1536)   │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1536)           │         6,144 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         6,148 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,795,827 (41.18 MB)

 Trainable params: 9,220 (36.02 KB)

 Non-trainable params: 10,786,607 (41.15 MB)

## 5) Phase 1 – Train with Frozen Base

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(LR_FROZEN),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_phase1 = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=8,   # was 5 — more room to converge
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1
    ),
    callbacks.ModelCheckpoint(
        CKPT_PHASE1, monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

print('=== PHASE 1: Frozen base (EfficientNetB3) ===')
print(f'Checkpoint → {CKPT_PHASE1}\n')

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN,
    class_weight=class_weight,
    callbacks=cb_phase1
)

=== PHASE 1: Frozen base (EfficientNetB3) ===
Checkpoint → /content/drive/MyDrive/BrainTumor_Results_v2/best_phase1.keras

Epoch 1/20
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.6009 - loss: 1.1109

KeyboardInterrupt: 

## 6) Phase 2 – Fine-tune Top Layers

Unfreeze the **top 20 layers** of EfficientNetB3 (v1 used 30 — too many, caused instability)  
and continue training at LR = 3e-5 (v1 used 1e-5 — too small to make progress).

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-FINETUNE_LAYERS]:   # top 20 only (was 30)
    layer.trainable = False

print(f'Trainable base layers: {sum(1 for l in base_model.layers if l.trainable)} / {len(base_model.layers)}')

model.compile(
    optimizer=keras.optimizers.Adam(LR_FINETUNE),     # 3e-5 (was 1e-5)
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_phase2 = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=10,   # was 7 — give fine-tuning more time
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1
    ),
    callbacks.ModelCheckpoint(
        CKPT_BEST, monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

print('=== PHASE 2: Fine-tuning (EfficientNetB3, top 20 layers) ===')
print(f'Best model → {CKPT_BEST}\n')

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN + EPOCHS_FINETUNE,
    class_weight=class_weight,
    callbacks=cb_phase2,
    initial_epoch=len(history1.history['loss'])
)

## 7) Training Curves

In [ ]:
def merge_history(h1, h2):
    return {k: h1.history[k] + h2.history[k] for k in h1.history}

hist         = merge_history(history1, history2)
epochs_ran   = range(1, len(hist['loss']) + 1)
phase2_start = len(history1.history['loss'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (train_k, val_k), title in zip(
    axes,
    [('loss', 'val_loss'), ('accuracy', 'val_accuracy')],
    ['Loss', 'Accuracy']
):
    ax.plot(epochs_ran, hist[train_k], label='Train')
    ax.plot(epochs_ran, hist[val_k],   label='Validation')
    ax.axvline(phase2_start, color='grey', linestyle='--', label='Fine-tune start')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()

plt.suptitle('Training Curves', fontsize=14)
plt.tight_layout()
savefig('training_curves.png')   # → Drive
plt.show()                        # → inline

## 8) Evaluate on Test Set

In [ ]:
best_model = keras.models.load_model(CKPT_BEST)

test_loss, test_acc = best_model.evaluate(test_ds, verbose=1)
print(f'\nTest Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

# Save to Drive
metrics_path = os.path.join(DRIVE_OUT, 'test_metrics.txt')
with open(metrics_path, 'w') as f:
    f.write(f'Test Loss:     {test_loss:.4f}\n')
    f.write(f'Test Accuracy: {test_acc:.4f}\n')
print(f'  💾 Saved → {metrics_path}')

## 9) Classification Report & Confusion Matrix

In [ ]:
y_true, y_pred_probs = [], []
for images, labels in test_ds:
    probs = best_model(images, training=False)
    y_true.extend(labels.numpy())
    y_pred_probs.extend(probs.numpy())

y_true       = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred       = np.argmax(y_pred_probs, axis=1)

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
print('Classification Report:')
print(report)

# Save to Drive
report_path = os.path.join(DRIVE_OUT, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write(report)
print(f'  💾 Saved → {report_path}')

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
)
plt.title('Confusion Matrix – Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
savefig('confusion_matrix.png')   # → Drive
plt.show()                         # → inline

## 10) Per-Class Prediction Confidence

In [ ]:
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(16, 4))
for i, cls in enumerate(CLASS_NAMES):
    mask = y_true == i
    axes[i].hist(y_pred_probs[mask, i], bins=20, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{cls}\n(n={mask.sum()})')
    axes[i].set_xlabel('Confidence')
    axes[i].set_xlim(0, 1)

axes[0].set_ylabel('Count')
plt.suptitle('Prediction Confidence (True Class Samples)', y=1.02)
plt.tight_layout()
savefig('confidence_distribution.png')   # → Drive
plt.show()                                # → inline

## 11) Sample Predictions

In [ ]:
n_show    = 12
sample_df = test_df.sample(n_show, random_state=SEED).reset_index(drop=True)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))

for i, row in sample_df.iterrows():
    img_raw     = tf.io.read_file(row['filepath'])
    img_decoded = tf.io.decode_jpeg(img_raw, channels=3)
    img_pre     = preprocess_input(
                    tf.cast(tf.image.resize(img_decoded, IMG_SIZE), tf.float32))

    prob     = best_model(tf.expand_dims(img_pre, 0), training=False).numpy()[0]
    pred_idx = int(np.argmax(prob))
    true_idx = CLASS_TO_IDX[row['label']]

    ax    = axes[i // 4][i % 4]
    color = 'green' if pred_idx == true_idx else 'red'
    ax.imshow(img_decoded.numpy().astype('uint8'), cmap='gray')
    ax.set_title(
        f'True: {row["label"]}\nPred: {CLASS_NAMES[pred_idx]} ({prob[pred_idx]:.2f})',
        color=color, fontsize=9
    )
    ax.axis('off')

plt.suptitle('Sample Predictions  (green = correct, red = wrong)', fontsize=13)
plt.tight_layout()
savefig('sample_predictions.png')   # → Drive
plt.show()                           # → inline

### Results and performance analysis
​
Phase 1 quickly reaches a validation accuracy around 0.85, already better than the initial epochs of v1. During Phase 2, fine‑tuning steadily improves validation accuracy, with the best checkpoint reaching roughly 0.92 validation accuracy and a validation loss around 0.22–0.27.
​

Training curves show that the model continues to learn during fine‑tuning but with diminishing returns after ~30–35 epochs, and later epochs mainly refine the decision boundary. On the whole, this EfficientNetB3 setup delivers solid performance, but the improvement over our lighter ResNet18 model is modest rather than dramatic.

## 12) Save Final Model & List Drive Outputs

In [ ]:
best_model.save(MODEL_FINAL)
print(f'✅ Final model saved → {MODEL_FINAL}')

print(f'\n📁 All files in {DRIVE_OUT}:')
for fname in sorted(os.listdir(DRIVE_OUT)):
    fpath   = os.path.join(DRIVE_OUT, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'   {fname:<48} {size_kb:>8.1f} KB')

---
## Summary

| Phase | Description | LR |
|---|---|---|
| 1 – Frozen | Train head only (EfficientNetB3 frozen) | 1e-3 |
| 2 – Fine-tune | Unfreeze top 20 layers of EfficientNetB3 | 3e-5 |

**Files saved to `My Drive/BrainTumor_Results_v2/`:**

| File | Description |
|---|---|
| `best_phase1.keras` | Best checkpoint from Phase 1 |
| `best_model.keras` | Best checkpoint overall (val accuracy) |
| `brain_tumor_classifier_v2.keras` | Final saved model |
| `training_curves.png` | Loss & accuracy over epochs |
| `confusion_matrix.png` | Per-class confusion matrix |
| `confidence_distribution.png` | Prediction confidence histograms |
| `sample_predictions.png` | 12 sample test predictions |
| `classification_report.txt` | Precision / Recall / F1 per class |
| `test_metrics.txt` | Final test loss & accuracy |



### **Why we did not adopt this solution**
​
Despite the respectable metrics, this approach is computationally heavy: EfficientNetB3 at 300×300 resolution, combined with strong data augmentation and two‑phase fine‑tuning, leads to long epoch times (on the order of several minutes per epoch and many dozens of epochs in total). This translates into higher GPU cost, slower experimentation cycles, and slower retraining when data or requirements change.
​

In addition, the accuracy gain compared with the ResNet18 model is limited; the more compact ResNet18 reaches similar or better performance with significantly faster training and inference. Given our constraints (limited compute budget, need for faster iterations, and deployment considerations), we chose not to move forward with this EfficientNetB3‑based solution and instead standardised on the more efficient ResNet18 model.